# WP44 — Ultraintelligence Trajectory (v0.4)
## CRLSSimulator · TrajectoryAnalyser · IntelligenceExplosionTest

Directly tests I.J. Good's (1965) empirical prediction: that a recursively
self-improving machine will show *accelerating* improvement — where each
generation improves the improvement procedure, not just performance.

We run the CRLS stack for 200 generations, fit three growth models (linear,
exponential, logistic), and perform a statistical test of Good's hypothesis:
**"Is accuracy gain correlated with the rate of change of the improvement
procedure itself?"**

> *"The first ultraintelligent machine … could design even better machines;
> there would then unquestionably be an 'intelligence explosion.'"*
> — I.J. Good (1965)

Runtime: **< 2 min** (pure Python, no GPU)

In [ ]:
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC'); sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path: sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import math, random, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (16, 6), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus; print(f'Prometheus {prometheus.__version__} OK')

In [ ]:
from prometheus.wp44_ultraintelligence_trajectory import (
    TrajectoryPoint, TrajectoryFit, IntelligenceExplosionTest,
    CRLSSimulator, TrajectoryAnalyser, TrajectoryReport,
    fit_linear, fit_exponential, fit_logistic,
    run_explosion_test, verify_wp44_exit_criteria,
)
from prometheus.wp17_crls_synthesis import SynthesisAction
print('WP44 imports OK')

In [ ]:
# ── Run the simulator for 200 generations
N_GEN = 200
analyser = TrajectoryAnalyser(
    n_generations    = N_GEN,
    initial_accuracy = 0.60,
    noise_std        = 0.012,
    seed             = 42,
)
report = analyser.run()
print(report.summary())

In [ ]:
# ── Action frequency breakdown
from collections import Counter
action_counts = Counter(p.action.name for p in report.points)
print('Synthesis action distribution:')
for action, count in sorted(action_counts.items()):
    pct = 100 * count / N_GEN
    bar = '#' * int(pct / 2)
    print(f'  {action:<20} {count:4d} ({pct:5.1f}%)  {bar}')

In [ ]:
# ── Intelligence explosion test detail
test = report.explosion_test
print('Intelligence Explosion Test (Good 1965)')
print('=' * 60)
print(test.hypothesis_text)
print()
print(f'  Pearson r    = {test.pearson_r:.4f}')
print(f'  p-value ≈    {test.p_value_approx:.4f}')
print(f'  n            = {test.n}')
print()
print(f'  Verdict: {test.verdict}')
print()
print(f'  {test.explanation}')

In [ ]:
gens      = [p.generation      for p in report.points]
accs      = [p.accuracy        for p in report.points]
gains     = [p.accuracy_gain   for p in report.points]
mg_norms  = [p.meta_grad_norm  for p in report.points]
p_safes   = [p.p_safe          for p in report.points]
entropies = [p.strategy_entropy for p in report.points]

action_colors = {
    'DEMOTE_WORST':  '#E53935',
    'PROMOTE_BEST':  '#43A047',
    'RESET_UNIFORM': '#FB8C00',
    'BOOST_UPWARD':  '#1E88E5',
}

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# ── Panel A: accuracy + three model fits
ax = axes[0, 0]
ax.plot(gens, accs, 'k-', lw=1, alpha=0.5, label='Observed accuracy')
lin_preds = report.fits['linear'].predictions
exp_preds = report.fits['exponential'].predictions
log_preds = report.fits['logistic'].predictions
ax.plot(gens, lin_preds, 'b--', lw=2, label=f"Linear  R²={report.fits['linear'].r_squared:.3f}")
ax.plot(gens, exp_preds, 'r-',  lw=2, label=f"Exp     R²={report.fits['exponential'].r_squared:.3f}")
ax.plot(gens, log_preds, 'g-',  lw=2, label=f"Logistic R²={report.fits['logistic'].r_squared:.3f}")
ax.set_xlabel('Generation'); ax.set_ylabel('Accuracy')
ax.set_title(f'Accuracy Trajectory\nBest fit: {report.best_fit.upper()}  '
             f'(Good verdict: {report.good_verdict})', fontweight='bold')
ax.legend(fontsize=9); ax.set_ylim(0, 1)

# ── Panel B: accuracy gain per generation
ax2 = axes[0, 1]
colors_g = [action_colors.get(p.action.name, 'grey') for p in report.points]
ax2.bar(gens, gains, color=colors_g, alpha=0.7, edgecolor='none')
ax2.axhline(0, color='black', lw=1)
# Smoothed trend
window = 10
smoothed = [sum(gains[max(0,i-window):i+1])/len(gains[max(0,i-window):i+1]) for i in range(len(gains))]
ax2.plot(gens, smoothed, 'k-', lw=2, label=f'{window}-gen moving avg')
ax2.set_xlabel('Generation'); ax2.set_ylabel('Accuracy gain Δa')
ax2.set_title('Per-Generation Accuracy Gain\n(coloured by synthesis action)', fontweight='bold')
patches = [mpatches.Patch(color=c, label=a) for a, c in action_colors.items()]
ax2.legend(handles=patches, fontsize=8, loc='upper left')

# ── Panel C: Good's test — scatter Δa vs |∇θ|
ax3 = axes[0, 2]
ax3.scatter(mg_norms[1:], gains[1:], c=colors_g[1:], alpha=0.6, s=20)
# Fit line
r = test.pearson_r
x_range = [min(mg_norms[1:]), max(mg_norms[1:])]
if abs(r) > 0.01:
    mn_x, mn_y = sum(mg_norms[1:])/len(mg_norms[1:]), sum(gains[1:])/len(gains[1:])
    sd_x = math.sqrt(sum((v-mn_x)**2 for v in mg_norms[1:])/len(mg_norms[1:]))
    sd_y = math.sqrt(sum((v-mn_y)**2 for v in gains[1:])/len(gains[1:]))
    if sd_x > 1e-10:
        slope  = r * sd_y / sd_x
        interc = mn_y - slope * mn_x
        ax3.plot(x_range, [slope*x+interc for x in x_range], 'r-', lw=2)
ax3.set_xlabel('Meta-gradient norm |∇θ|'); ax3.set_ylabel('Accuracy gain Δa')
verdict_color = {'SUPPORTED': '#43A047', 'INCONCLUSIVE': '#FB8C00', 'NOT_SUPPORTED': '#E53935'}
ax3.set_title(
    f"Good's Hypothesis Test\nr={test.pearson_r:.3f}  p≈{test.p_value_approx:.3f}  "
    f"[{test.verdict}]",
    fontweight='bold',
    color=verdict_color.get(test.verdict, 'black')
)

# ── Panel D: meta-gradient norm over time
ax4 = axes[1, 0]
ax4.plot(gens, mg_norms, color='#7B1FA2', lw=1, alpha=0.7)
sm_mg = [sum(mg_norms[max(0,i-10):i+1])/len(mg_norms[max(0,i-10):i+1]) for i in range(len(mg_norms))]
ax4.plot(gens, sm_mg, 'k-', lw=2, label='10-gen MA')
ax4.set_xlabel('Generation'); ax4.set_ylabel('|∇θ| (meta-gradient norm)')
ax4.set_title('WP21 Meta-Gradient Norm\n(rate of improvement-procedure change)', fontweight='bold')
ax4.legend()

# ── Panel E: WP40 safety gate p_safe over time
ax5 = axes[1, 1]
ax5.plot(gens, p_safes, color='#00796B', lw=1, alpha=0.7)
ax5.axhline(0.60, color='red', linestyle='--', lw=1.5, label='Safety threshold 0.60')
blocked = [g for g, p in zip(gens, report.points) if p.action == SynthesisAction.RESET_UNIFORM
           and report.points[g].p_safe < 0.60]
if blocked:
    ax5.scatter(blocked, [p_safes[g] for g in blocked], color='red', s=20, zorder=5, label='Gated to RESET')
ax5.set_xlabel('Generation'); ax5.set_ylabel('P(safe)')
ax5.set_title('WP40 Safety Gate: P(safe) per Generation', fontweight='bold')
ax5.legend(fontsize=9); ax5.set_ylim(0, 1)

# ── Panel F: strategy entropy over time
ax6 = axes[1, 2]
ax6.plot(gens, entropies, color='#F57C00', lw=1, alpha=0.7)
H_max = math.log(4)
ax6.axhline(H_max, color='blue', linestyle=':', lw=1.5, label=f'H_max (uniform) = {H_max:.2f}')
ax6.set_xlabel('Generation'); ax6.set_ylabel('Strategy entropy (nats)')
ax6.set_title('Strategy Distribution Entropy\n(diversity of synthesis actions)', fontweight='bold')
ax6.legend(fontsize=9)

fig.suptitle('WP44: Ultraintelligence Trajectory — Testing Good (1965) Empirically',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('wp44_ultraintelligence_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved wp44_ultraintelligence_trajectory.png')

In [ ]:
criteria = verify_wp44_exit_criteria(analyser, report)
print('WP44 Exit Criteria Verification'); print('=' * 62)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
if all(criteria.values()):
    print()
    print('All WP44 exit criteria satisfied.')
    print()
    print(f"Good's Verdict: {report.good_verdict}")
    print(f"  {report.good_statement}")

---
## Conclusions

**WP44** provides the first direct empirical test of Good's intelligence explosion
hypothesis within Prometheus:

| Model | R² | Interpretation |
|-------|----|----------------|
| Linear | see output | Baseline: steady but not accelerating |
| Exponential | see output | Good's prediction: accelerating returns |
| Logistic | see output | Bounded explosion: ceiling effect |

### Key findings

1. **Accuracy trajectory**: The best-fit model reveals whether improvement is
   linear, super-linear (explosion), or saturating.

2. **Good's test**: The Pearson correlation between accuracy gain Δa and
   meta-gradient norm |∇θ| tests the core claim: *does each generation improve
   the improvement procedure?*  A positive r with p < 0.05 supports H₁.

3. **WP40 safety gating**: The safety gate engages when p_safe < 0.60, forcing
   RESET_UNIFORM — demonstrating that the Gödelian governor actively shapes the
   trajectory, not just monitors it.

4. **Strategy entropy**: Low entropy (one action dominating) triggers RESET_UNIFORM;
   high entropy allows exploration — the classic exploration-exploitation tension
   plays out across 200 generations.

### References
- Good (1965) — "Speculations Concerning the First Ultraintelligent Machine"
- Schmidhuber (2007) — "Gödel Machines"
- Silver et al. (2017) — AlphaZero super-linear improvement curves